# ЛАБОРАТОРНА РОБОТА 4

In [43]:
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.animation as animation

## Мурашиний алгоритм

$$
\eta_{ij} = \frac{1}{D_{ij}}
$$

$$
P_{ij,k}(t) =
\begin{cases}
\frac{[\tau_{ij}(t)]^{\alpha} [\eta_{ij}]^{\beta}}
{\sum\limits_{l \in J_{i,k}} [\tau_{il}(t)]^{\alpha} [\eta_{il}]^{\beta}}, & j \in J_{i,k} \\\\
0, & j \notin J_{i,k}
\end{cases}
$$

$$
\Delta \tau_{ij,k}(t) =
\begin{cases}
\frac{Q}{L_k(t)}, & (i,j) \in T_k(t) \\\\
0, & (i,j) \notin T_k(t)
\end{cases}
$$

$$
\tau_{ij}(t+1) = (1 - \rho)\tau_{ij}(t) + \Delta \tau_{ij}(t)
$$

In [44]:
ants = 20
iterations = 70

In [ ]:
def ant_algorithm(dist_matrix, alpha = 1.3, beta = 6, rho = 0.55, Q = 150):
    
    n_cities = dist_matrix.shape[0]
    tau = np.ones((n_cities, n_cities))
    eta = 1 / (dist_matrix + np.diag([np.inf]*n_cities))
    
    best_path = None  
    best_length = np.inf

    best_history = []
    best_path_history = []
    
    for i in range(iterations):
        
        all_paths = []
        all_lengths = []
        
        for ant in range(ants):
            
            start = np.random.randint(n_cities)
            path = [start]
            visited = set(path)
            
            while len(path) < n_cities:
                
                current = path[-1]
                
                allowed_cities = []
                
                for city in range(n_cities):
                    if city not in visited:
                        allowed_cities.append(city)
                
                probability = np.array([
                    (tau[current, j]**alpha) * (eta[current, j]**beta)
                    for j in allowed_cities
                ])
                probability /= probability.sum()
                
                next_city = np.random.choice(allowed_cities, p = probability)
                path.append(next_city)
                visited.add(next_city)
            
            length = sum(dist_matrix[path[i], path[i+1]] for i in range(n_cities-1))
            length += dist_matrix[path[-1], path[0]]
            
            all_paths.append(path)
            all_lengths.append(length)
            
            if length < best_length:
                best_length = length
                best_path = path.copy()  
                
        delta_tau = np.zeros((n_cities, n_cities))
        
        for path, length in zip(all_paths, all_lengths):
            for i in range(n_cities):
                j = (i + 1) % n_cities
                delta_tau[path[i], path[j]] += Q / length
        
        tau = (1 - rho) * tau + delta_tau

        best_history.append(best_length)
        best_path_history.append(best_path.copy())
    
    return best_path, best_length, best_history, best_path_history

## Генетичний алгоритм

In [46]:
pop_size = 50 
generations = 400

In [ ]:
def create_individual(n_cities):
    
    ind = list(range(n_cities))
    random.shuffle(ind)
    return ind

def fitness(ind, dist_matrix):
    
    total_distance = 0  
    for i in range(len(ind) - 1):
        total_distance += dist_matrix[ind[i] , ind[i + 1]]
        
    total_distance += dist_matrix[ind[-1], ind[0]]

    return total_distance

def selection(population, dist_matrix):
    
    candidates = random.sample(population, 3)
    return min(candidates, key=lambda ind: fitness(ind, dist_matrix))


def crossover(parent1, parent2):
    
    size = len(parent1)
    start, end = sorted(random.sample(range(size), 2))
    child = [-1] * size
    child[start:end] = parent1[start:end]
    fill = [x for x in parent2 if x not in child]

    idx = 0
    for i in range(size):
        if child[i] == -1:
            child[i] = fill[idx]
            idx += 1

    return child

def mutate(ind):
    
    i, j = sorted(random.sample(range(len(ind)), 2))
    ind[i:j] = reversed(ind[i:j])

In [ ]:
def ga_algorithm(dist_matrix, mutation_prob = 0.2):
    
    n_cities = dist_matrix.shape[0]
    
    population = [create_individual(n_cities) for _ in range(pop_size)]

    best_ind = None
    best_len = float("inf")

    best_history = []
    best_path_history = []

    for _ in range(generations):

        population.sort(key=lambda ind: fitness(ind, dist_matrix))

        new_population = population[:1]

        current_best = population[0]
        current_len = fitness(current_best, dist_matrix)

        if current_len < best_len:
            best_len = current_len
            best_ind = current_best.copy()

        best_history.append(best_len)
        best_path_history.append(best_ind.copy())

        while len(new_population) < pop_size:

            parent1 = selection(population, dist_matrix)
            parent2 = selection(population, dist_matrix)

            child = crossover(parent1, parent2)

            if random.random() < mutation_prob:
                mutate(child)

            new_population.append(child)

        population = new_population

    return best_ind, best_len, best_history, best_path_history

In [49]:
def create_gif(points, best_paths_history, best_history, name, filename):
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    ax_route, ax_conv = axes

    suptitle = fig.suptitle("", fontsize=14, fontweight="bold")

    ymin = np.min(best_history)
    ymax = np.max(best_history)

    frame_indices = list(range(0, len(best_paths_history), 5))

    def update(frame_idx):
        frame = frame_indices[frame_idx]

        ax_route.cla()
        ax_conv.cla()

        path = best_paths_history[frame]

        ordered = points[path + [path[0]]]

        ax_route.plot(ordered[:, 0], ordered[:, 1], '-k', lw=1)
        ax_route.scatter(ordered[:, 0], ordered[:, 1], color='red', s=50, zorder=5)

        ax_route.set_title(f"Маршрут (ітерація {frame})")
        ax_route.axis('equal')
        ax_route.grid()

        ax_conv.plot(best_history[:frame+1], lw=2)
        ax_conv.set_title("Збіжність")
        ax_conv.set_xlabel("Ітерація")
        ax_conv.set_ylabel("Довжина")
        ax_conv.set_xlim(0, len(best_history))
        ax_conv.set_ylim(ymin, ymax)
        ax_conv.grid()

        suptitle.set_text(
            f"{name}\nНайкраща довжина: {best_history[frame]:.4f}"
        )

        return ax_route, ax_conv

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(frame_indices),
        interval=400,
        repeat=False
    )

    ani.save(filename, writer="pillow")
    plt.close(fig)

## Генерування тестових даних

In [ ]:
def generate_data(n_points, radius_min=5, radius_max=10, center=(0, 0)):

    angles = np.linspace(0, 2*np.pi, n_points, endpoint=False)
    radius = np.random.uniform(radius_min, radius_max, n_points)
    
    points = np.zeros((n_points, 2))
    points[:, 0] = center[0] + radius * np.cos(angles)
    points[:, 1] = center[1] + radius * np.sin(angles)

    dist = np.zeros((n_points, n_points))
    for i in range(n_points):
        for j in range(n_points):
            dist[i, j] = np.linalg.norm(points[i] - points[j])

    return points, dist

## Результат роботи кожного з двох алгоритмів на 30, 50, 100 та 200 точоках

In [51]:
sizes = [30, 50, 100, 200]

for n in sizes:
    print(f"\n------ {n} точок ------")

    points, dist = generate_data(n)

    ga_path, ga_len, ga_hist, ga_path_hist = ga_algorithm(dist)
    aco_path, aco_len, aco_hist, aco_path_hist = ant_algorithm(dist)

    print(f"GA: {ga_len:.4f}")
    print(f"ANT: {aco_len:.4f}")

    create_gif(points, ga_path_hist, ga_hist, name=f"GA (n={n})", filename=f"ga_{n}.gif")
    create_gif(points, aco_path_hist, aco_hist, name=f"ANT (n={n})", filename=f"ant_{n}.gif")


------ 30 точок ------
GA: 72.9197
ANT: 73.0557

------ 50 точок ------
GA: 98.5654
ANT: 86.4577

------ 100 точок ------
GA: 186.1395
ANT: 135.9798

------ 200 точок ------
GA: 494.6069
ANT: 204.2420


## Результат для готових датасетів

In [ ]:
def read_tsp(filename):

    points = []
    
    with open(filename, 'r') as f:
        
        lines = f.readlines()
        read_nodes = False
        
        for line in lines:
            
            line = line.strip()
            if line.startswith("NODE_COORD_SECTION"):
                read_nodes = True
                continue
            
            if line.startswith("EOF"):
                break
            
            if read_nodes:
                
                parts = line.split()
                x, y = float(parts[1]), float(parts[2])
                points.append([x, y])
                    
    return np.array(points)

def dist_matrix(points):
    
    n = len(points)
    dist_matrix = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            dist_matrix[i, j] = np.linalg.norm(points[i] - points[j])
                
    return dist_matrix

In [ ]:
data = ["dataset1.tsp", "dataset2.tsp" ]

for d in data:
    print(f"\n------ {d}  ------")

    points = read_tsp(d)
    dist = dist_matrix(points)

    ga_path, ga_len, ga_hist, ga_path_hist = ga_algorithm(dist)
    aco_path, aco_len, aco_hist, aco_path_hist = ant_algorithm(dist)
          
    print(f"GA: {ga_len:.4f}")
    print(f"ANT: {aco_len:.4f}")

    create_gif(points, ga_path_hist, ga_hist, name=f"GA {d}", filename=f"ga_{d}.gif")
    create_gif(points, aco_path_hist, aco_hist, name=f"ANT {d}", filename=f"ant_{d}.gif")


------ dataset1.tsp  ------
GA: 909.2996
ANT: 665.5559

------ dataset2.tsp  ------
GA: 6854.1663
ANT: 1518.4597
